# Category 7 — Advanced Window Functions
### Total Questions: 11

---

1. Calculate the running total of salaries within each department
2. Calculate the total amount spent and the cumulative total amount spent over time for each user
3. Calculate the running total of the amount spent by all users over time
4. Find the percentage of total salary for each employee within their department
5. Use LAG function to find the previous employee's salary within each department
6. Use LEAD function to find the next employee's salary within each department
7. Find all dates with higher temperature compared to its previous date (LAG approach)
8. Find all dates with higher temperature compared to its previous date (Self Join approach)
9. Write a SQL query to find rolling average of posts on daily basis for each user_id
10. Calculate the rolling average of order amounts over time
11. Calculate the running total of order amounts over time

# ⬆️ Upload sql_practice.db from your computer


In [ ]:
# ⬆️ Upload sql_practice.db from your computer
from google.colab import files
uploaded = files.upload()
print("✅ Database uploaded!")

Saving sql_practice.db to sql_practice.db
✅ Database uploaded!


In [ ]:
import sqlite3
import pandas as pd
import os

_cwd = os.getcwd()
_db = os.path.join(_cwd, "sql_practice.db")
if not os.path.isfile(_db):
    _db = os.path.join(os.path.dirname(_cwd), "sql_practice.db")
conn = sqlite3.connect(_db)

# Verify tables loaded
pd.read_sql_query("""
SELECT name FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)


,name
0,customers
1,department
2,emp
3,employee_salary
4,employees
5,monthly_salary
6,num
7,orders
8,posts
9,products


## 1. Running Total of Salaries Within Each Department

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / SUM OVER + PARTITION BY

---

**Table: employee_salary**
```
+------------------+---------+
| Column Name      | Type    |
+------------------+---------+
| did              | int     |
| name_of_employee | varchar |
| salary           | int     |
+------------------+---------+
```

- `did` represents the department ID.
- `name_of_employee` represents the employee's name.
- `salary` represents the employee's salary.

Each department may contain multiple employees.

---

**Problem**

Write a SQL query to calculate the **running total of salaries** within each department.

The running total should reset for each new department.

Return `did`, `name_of_employee`, `salary` and `running_total`.

**Example 1**

**Input**

employee_salary table:
```
+-----+------------------+--------+
| did | name_of_employee | salary |
+-----+------------------+--------+
| 1   | Alice            | 50000  |
| 1   | Bob              | 65000  |
| 1   | Carol            | 60000  |
| 2   | David            | 70000  |
| 2   | Emma             | 72000  |
| 2   | Frank            | 68000  |
| 3   | Anna             | 55000  |
| 3   | George           | 52000  |
| 3   | Henry            | 51000  |
+-----+------------------+--------+
```

**Output**
```
+-----+------------------+--------+---------------+
| did | name_of_employee | salary | running_total |
+-----+------------------+--------+---------------+
| 1   | Alice            | 50000  | 50000         |
| 1   | Bob              | 65000  | 115000        |
| 1   | Carol            | 60000  | 175000        |
| 2   | David            | 70000  | 70000         |
| 2   | Emma             | 72000  | 142000        |
| 2   | Frank            | 68000  | 210000        |
| 3   | Anna             | 55000  | 55000         |
| 3   | George           | 52000  | 107000        |
| 3   | Henry            | 51000  | 158000        |
+-----+------------------+--------+---------------+
```

CODE (View Table):

In [ ]:
# 👀 Preview the table
print("employee_salary table:")
pd.read_sql_query("SELECT * FROM employee_salary", conn)

employee_salary table:


,did,name_of_employee,salary
0,1,Alice,50000
1,1,Bob,65000
2,1,Carol,60000
3,2,David,70000
4,2,Emma,72000
5,2,Frank,68000
6,3,Anna,55000
7,3,George,52000
8,3,Henry,51000


# ✅ Solution


In [ ]:
# ✅ Solution
query = """
SELECT did,
       name_of_employee,
       salary,
       SUM(salary) OVER (PARTITION BY did ORDER BY salary) AS running_total
FROM employee_salary
"""

pd.read_sql_query(query, conn)

,did,name_of_employee,salary,running_total
0,1,Alice,50000,50000
1,1,Carol,60000,110000
2,1,Bob,65000,175000
3,2,Frank,68000,68000
4,2,David,70000,138000
5,2,Emma,72000,210000
6,3,Henry,51000,51000
7,3,George,52000,103000
8,3,Anna,55000,158000


**Explanation**

- `SUM(salary) OVER (...)` is a window function that calculates a running total.
- `PARTITION BY did` resets the running total for each new department.
- `ORDER BY salary` defines the order in which salaries are accumulated row by row.
- Unlike `GROUP BY`, window functions do **not collapse rows** — every row is returned with its running total.

## 2. Total and Cumulative Amount Spent Over Time Per User

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / SUM OVER + PARTITION BY + ORDER BY DATE

---

**Table: zomato_orders**
```
+---------------------+---------+
| Column Name         | Type    |
+---------------------+---------+
| user_id             | int     |
| order_id            | int     |
| amount              | int     |
| order_date          | date    |
| delivery_partner_id | int     |
| delivery_rating     | int     |
+---------------------+---------+
```

- `user_id` represents the user who placed the order.
- `amount` represents the order amount.
- `order_date` represents when the order was placed.

---

**Problem**

Write a SQL query to calculate the **total amount spent** and the **cumulative total amount spent over time** for each user.

Return `user_id`, `order_date`, `amount` and `cumulative_total`.

**Example 1**

**Input**

zomato_orders table:
```
+---------+----------+--------+------------+---------------------+-----------------+
| user_id | order_id | amount | order_date | delivery_partner_id | delivery_rating |
+---------+----------+--------+------------+---------------------+-----------------+
| 1       | 101      | 200    | 2024-01-01 | 1                   | 5               |
| 1       | 102      | 300    | 2024-01-03 | 2                   | 4               |
| 2       | 103      | 150    | 2024-01-04 | 1                   | 5               |
| 3       | 104      | 400    | 2024-01-05 | 3                   | 3               |
+---------+----------+--------+------------+---------------------+-----------------+
```

**Output**
```
+---------+------------+--------+-----------------+
| user_id | order_date | amount | cumulative_total |
+---------+------------+--------+-----------------+
| 1       | 2024-01-01 | 200    | 200             |
| 1       | 2024-01-03 | 300    | 500             |
| 2       | 2024-01-04 | 150    | 150             |
| 3       | 2024-01-05 | 400    | 400             |
+---------+------------+--------+-----------------+
```

CODE (View Table):

In [ ]:
# 👀 Preview the table
print("zomato_orders table:")
pd.read_sql_query("SELECT * FROM zomato_orders", conn)

zomato_orders table:


,user_id,order_id,amount,order_date,delivery_partner_id,delivery_rating
0,1,101,200,2024-01-01,1,5
1,1,102,300,2024-01-03,2,4
2,2,103,150,2024-01-04,1,5
3,3,104,400,2024-01-05,3,3


CODE (Solution):

In [ ]:
# ✅ Solution
query = """
SELECT user_id,
       order_date,
       amount,
       SUM(amount) OVER (PARTITION BY user_id ORDER BY order_date) AS cumulative_total
FROM zomato_orders
"""

pd.read_sql_query(query, conn)

,user_id,order_date,amount,cumulative_total
0,1,2024-01-01,200,200
1,1,2024-01-03,300,500
2,2,2024-01-04,150,150
3,3,2024-01-05,400,400


**Explanation**

- `PARTITION BY user_id` resets the cumulative total for each user separately.
- `ORDER BY order_date` accumulates the amount in chronological order.
- Each row shows how much a user has spent **up to and including that order date**.
- User 1 spent 200 on Jan 1 → cumulative is 200. Then spent 300 on Jan 3 → cumulative becomes 500.

## 3. Running Total of Amount Spent by All Users Over Time

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / SUM OVER + ORDER BY DATE (No Partition)

---

**Table: zomato_orders**
```
+---------------------+---------+
| Column Name         | Type    |
+---------------------+---------+
| user_id             | int     |
| order_id            | int     |
| amount              | int     |
| order_date          | date    |
| delivery_partner_id | int     |
| delivery_rating     | int     |
+---------------------+---------+
```

- `user_id` represents the user who placed the order.
- `amount` represents the order amount.
- `order_date` represents when the order was placed.

---

**Problem**

Write a SQL query to calculate the **running total of the amount spent by ALL users combined** over time.

Return `user_id`, `order_date`, `amount` and `running_total`.

**Example 1**

**Input**

zomato_orders table:
```
+---------+----------+--------+------------+---------------------+-----------------+
| user_id | order_id | amount | order_date | delivery_partner_id | delivery_rating |
+---------+----------+--------+------------+---------------------+-----------------+
| 1       | 101      | 200    | 2024-01-01 | 1                   | 5               |
| 1       | 102      | 300    | 2024-01-03 | 2                   | 4               |
| 2       | 103      | 150    | 2024-01-04 | 1                   | 5               |
| 3       | 104      | 400    | 2024-01-05 | 3                   | 3               |
+---------+----------+--------+------------+---------------------+-----------------+
```

**Output**
```
+---------+------------+--------+---------------+
| user_id | order_date | amount | running_total |
+---------+------------+--------+---------------+
| 1       | 2024-01-01 | 200    | 200           |
| 1       | 2024-01-03 | 300    | 500           |
| 2       | 2024-01-04 | 150    | 650           |
| 3       | 2024-01-05 | 400    | 1050          |
+---------+------------+--------+---------------+
```

# 👀 Preview the table

In [ ]:
# 👀 Preview the table
print("zomato_orders table:")
pd.read_sql_query("SELECT * FROM zomato_orders", conn)

zomato_orders table:


,user_id,order_id,amount,order_date,delivery_partner_id,delivery_rating
0,1,101,200,2024-01-01,1,5
1,1,102,300,2024-01-03,2,4
2,2,103,150,2024-01-04,1,5
3,3,104,400,2024-01-05,3,3


# ✅ Solution

In [ ]:
# ✅ Solution
query = """
SELECT user_id,
       order_date,
       amount,
       SUM(amount) OVER (ORDER BY order_date) AS running_total
FROM zomato_orders
"""

pd.read_sql_query(query, conn)

,user_id,order_date,amount,running_total
0,1,2024-01-01,200,200
1,1,2024-01-03,300,500
2,2,2024-01-04,150,650
3,3,2024-01-05,400,1050


**Explanation**

- No `PARTITION BY` here — the running total is calculated **across all users globally**.
- `ORDER BY order_date` accumulates the amount chronologically across the entire table.
- Compare with Q2 — Q2 resets per user, Q3 never resets and keeps growing across all orders.
- By Jan 5 the total across all users is 200 + 300 + 150 + 400 = 1050.

## 4. Percentage of Total Salary for Each Employee Within Their Department

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / SUM OVER + PARTITION BY + Arithmetic

---

**Table: employee_salary**
```
+------------------+---------+
| Column Name      | Type    |
+------------------+---------+
| did              | int     |
| name_of_employee | varchar |
| salary           | int     |
+------------------+---------+
```

- `did` represents the department ID.
- `name_of_employee` represents the employee's name.
- `salary` represents the employee's salary.

---

**Problem**

Write a SQL query to find the **percentage contribution** of each employee's salary to their department's total salary.

Return `did`, `name_of_employee`, `salary` and `salary_percentage`.

**Example 1**

**Input**

employee_salary table:
```
+-----+------------------+--------+
| did | name_of_employee | salary |
+-----+------------------+--------+
| 1   | Alice            | 50000  |
| 1   | Bob              | 65000  |
| 1   | Carol            | 60000  |
| 2   | David            | 70000  |
| 2   | Emma             | 72000  |
| 2   | Frank            | 68000  |
| 3   | Anna             | 55000  |
| 3   | George           | 52000  |
| 3   | Henry            | 51000  |
+-----+------------------+--------+
```

**Output**
```
+-----+------------------+--------+--------------------+
| did | name_of_employee | salary | salary_percentage  |
+-----+------------------+--------+--------------------+
| 1   | Alice            | 50000  | 28.57              |
| 1   | Bob              | 65000  | 37.14              |
| 1   | Carol            | 60000  | 34.29              |
| 2   | David            | 70000  | 33.33              |
| 2   | Emma             | 72000  | 34.29              |
| 2   | Frank            | 68000  | 32.38              |
| 3   | Anna             | 55000  | 34.38              |
| 3   | George           | 52000  | 32.50              |
| 3   | Henry            | 51000  | 31.88              |
+-----+------------------+--------+--------------------+
```

# 👀 Preview the table

In [ ]:
# 👀 Preview the table
print("employee_salary table:")
pd.read_sql_query("SELECT * FROM employee_salary", conn)

employee_salary table:


,did,name_of_employee,salary
0,1,Alice,50000
1,1,Bob,65000
2,1,Carol,60000
3,2,David,70000
4,2,Emma,72000
5,2,Frank,68000
6,3,Anna,55000
7,3,George,52000
8,3,Henry,51000


# ✅ Solution

In [ ]:
# ✅ Solution
query = """
SELECT did,
       name_of_employee,
       salary,
       ROUND(salary * 100.0 / SUM(salary) OVER (PARTITION BY did), 2) AS salary_percentage
FROM employee_salary
"""

pd.read_sql_query(query, conn)

,did,name_of_employee,salary,salary_percentage
0,1,Alice,50000,28.57
1,1,Bob,65000,37.14
2,1,Carol,60000,34.29
3,2,David,70000,33.33
4,2,Emma,72000,34.29
5,2,Frank,68000,32.38
6,3,Anna,55000,34.81
7,3,George,52000,32.91
8,3,Henry,51000,32.28


**Explanation**

- `SUM(salary) OVER (PARTITION BY did)` calculates the **total salary of the entire department** for each row.
- `salary * 100.0 / SUM(salary) OVER (PARTITION BY did)` divides each employee's salary by the department total and multiplies by 100 to get the percentage.
- `ROUND(..., 2)` rounds the result to 2 decimal places.
- We use `100.0` instead of `100` to force decimal division in SQLite.

## 5. LAG — Previous Employee's Salary Within Each Department

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / LAG + PARTITION BY

---

**Table: employee_salary**
```
+------------------+---------+
| Column Name      | Type    |
+------------------+---------+
| did              | int     |
| name_of_employee | varchar |
| salary           | int     |
+------------------+---------+
```

- `did` represents the department ID.
- `name_of_employee` represents the employee's name.
- `salary` represents the employee's salary.

---

**Problem**

Write a SQL query to use the **LAG function** to find the **previous employee's salary** within each department.

If there is no previous employee the value should be `NULL`.

Return `did`, `name_of_employee`, `salary` and `prev_salary`.

**Example 1**

**Input**

employee_salary table:
```
+-----+------------------+--------+
| did | name_of_employee | salary |
+-----+------------------+--------+
| 1   | Alice            | 50000  |
| 1   | Bob              | 65000  |
| 1   | Carol            | 60000  |
| 2   | David            | 70000  |
| 2   | Emma             | 72000  |
| 2   | Frank            | 68000  |
| 3   | Anna             | 55000  |
| 3   | George           | 52000  |
| 3   | Henry            | 51000  |
+-----+------------------+--------+
```

**Output**
```
+-----+------------------+--------+-------------+
| did | name_of_employee | salary | prev_salary |
+-----+------------------+--------+-------------+
| 1   | Alice            | 50000  | NULL        |
| 1   | Bob              | 65000  | 50000       |
| 1   | Carol            | 60000  | 65000       |
| 2   | David            | 70000  | NULL        |
| 2   | Emma             | 72000  | 70000       |
| 2   | Frank            | 68000  | 72000       |
| 3   | Anna             | 55000  | NULL        |
| 3   | George           | 52000  | 55000       |
| 3   | Henry            | 51000  | 52000       |
+-----+------------------+--------+-------------+
```

# 👀 Preview the table

In [ ]:
# 👀 Preview the table
print("employee_salary table:")
pd.read_sql_query("SELECT * FROM employee_salary", conn)

employee_salary table:


,did,name_of_employee,salary
0,1,Alice,50000
1,1,Bob,65000
2,1,Carol,60000
3,2,David,70000
4,2,Emma,72000
5,2,Frank,68000
6,3,Anna,55000
7,3,George,52000
8,3,Henry,51000


# ✅ Solution

In [ ]:
# ✅ Solution
query = """
SELECT did,
       name_of_employee,
       salary,
       LAG(salary) OVER (PARTITION BY did ORDER BY salary) AS prev_salary
FROM employee_salary
"""

pd.read_sql_query(query, conn)

,did,name_of_employee,salary,prev_salary
0,1,Alice,50000,NaN
1,1,Carol,60000,50000.0
2,1,Bob,65000,60000.0
3,2,Frank,68000,NaN
4,2,David,70000,68000.0
5,2,Emma,72000,70000.0
6,3,Henry,51000,NaN
7,3,George,52000,51000.0
8,3,Anna,55000,52000.0


**Explanation**

- `LAG(salary)` fetches the value of `salary` from the **previous row** within the window.
- `PARTITION BY did` resets the LAG for each department — so the first employee in each department always gets `NULL`.
- `ORDER BY salary` defines the order in which rows are processed.
- LAG is very useful for **comparing current row with previous row** — like finding salary differences or temperature changes.

## 6. LEAD — Next Employee's Salary Within Each Department

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / LEAD + PARTITION BY

---

**Table: employee_salary**
```
+------------------+---------+
| Column Name      | Type    |
+------------------+---------+
| did              | int     |
| name_of_employee | varchar |
| salary           | int     |
+------------------+---------+
```

- `did` represents the department ID.
- `name_of_employee` represents the employee's name.
- `salary` represents the employee's salary.

---

**Problem**

Write a SQL query to use the **LEAD function** to find the **next employee's salary** within each department.

If there is no next employee the value should be `NULL`.

Return `did`, `name_of_employee`, `salary` and `next_salary`.

**Example 1**

**Input**

employee_salary table:
```
+-----+------------------+--------+
| did | name_of_employee | salary |
+-----+------------------+--------+
| 1   | Alice            | 50000  |
| 1   | Bob              | 65000  |
| 1   | Carol            | 60000  |
| 2   | David            | 70000  |
| 2   | Emma             | 72000  |
| 2   | Frank            | 68000  |
| 3   | Anna             | 55000  |
| 3   | George           | 52000  |
| 3   | Henry            | 51000  |
+-----+------------------+--------+
```

**Output**
```
+-----+------------------+--------+-------------+
| did | name_of_employee | salary | next_salary |
+-----+------------------+--------+-------------+
| 1   | Alice            | 50000  | 65000       |
| 1   | Bob              | 65000  | 60000       |
| 1   | Carol            | 60000  | NULL        |
| 2   | David            | 70000  | 72000       |
| 2   | Emma             | 72000  | 68000       |
| 2   | Frank            | 68000  | NULL        |
| 3   | Anna             | 55000  | 52000       |
| 3   | George           | 52000  | 51000       |
| 3   | Henry            | 51000  | NULL        |
+-----+------------------+--------+-------------+
```

# 👀 Preview the table

In [ ]:
# 👀 Preview the table
print("employee_salary table:")
pd.read_sql_query("SELECT * FROM employee_salary", conn)

employee_salary table:


,did,name_of_employee,salary
0,1,Alice,50000
1,1,Bob,65000
2,1,Carol,60000
3,2,David,70000
4,2,Emma,72000
5,2,Frank,68000
6,3,Anna,55000
7,3,George,52000
8,3,Henry,51000


# ✅ Solution


In [ ]:
# ✅ Solution
query = """
SELECT did,
       name_of_employee,
       salary,
       LEAD(salary) OVER (PARTITION BY did ORDER BY salary) AS next_salary
FROM employee_salary
"""

pd.read_sql_query(query, conn)

,did,name_of_employee,salary,next_salary
0,1,Alice,50000,60000.0
1,1,Carol,60000,65000.0
2,1,Bob,65000,NaN
3,2,Frank,68000,70000.0
4,2,David,70000,72000.0
5,2,Emma,72000,NaN
6,3,Henry,51000,52000.0
7,3,George,52000,55000.0
8,3,Anna,55000,NaN


**Explanation**

- `LEAD(salary)` fetches the value of `salary` from the **next row** within the window.
- `PARTITION BY did` resets the LEAD for each department — so the last employee in each department always gets `NULL`.
- `ORDER BY salary` defines the order in which rows are processed.
- LEAD is the **opposite of LAG** — LAG looks back, LEAD looks forward.
- Together LAG and LEAD are used to compare adjacent rows without a self join.

## 7. Dates With Higher Temperature Than Previous Date

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / LAG + DATE Comparison

---

**Table: temperature**
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| days        | date    |
| temp        | int     |
+-------------+---------+
```

- `days` represents the date.
- `temp` represents the temperature recorded on that date.

---

**Problem**

Write a SQL query to find all **dates where the temperature was higher than the previous day's temperature**.

Return the `days` column only.

**Example 1**

**Input**

temperature table:
```
+------------+------+
| days       | temp |
+------------+------+
| 2024-01-01 | 20   |
| 2024-01-02 | 22   |
| 2024-01-03 | 21   |
| 2024-01-04 | 25   |
| 2024-01-05 | 26   |
+------------+------+
```

**Output**
```
+------------+
| days       |
+------------+
| 2024-01-02 |
| 2024-01-04 |
| 2024-01-05 |
+------------+
```

# 👀 Preview the table

In [ ]:
# 👀 Preview the table
print("temperature table:")
pd.read_sql_query("SELECT * FROM temperature", conn)

temperature table:


,days,temp
0,2024-01-01,20
1,2024-01-02,22
2,2024-01-03,21
3,2024-01-04,25
4,2024-01-05,26


# ✅ Solution using LAG

In [ ]:
# ✅ Solution using LAG
query = """
SELECT days
FROM (
    SELECT days,
           temp,
           LAG(temp) OVER (ORDER BY days) AS prev_temp
    FROM temperature
)
WHERE temp > prev_temp
"""

pd.read_sql_query(query, conn)

,days
0,2024-01-02
1,2024-01-04
2,2024-01-05


**Explanation**

- `LAG(temp) OVER (ORDER BY days)` fetches the temperature from the **previous date**.
- The inner subquery adds a `prev_temp` column to every row.
- The outer `WHERE temp > prev_temp` keeps only those dates where today's temperature is **higher than yesterday's**.
- Jan 2 (22 > 20) ✅, Jan 3 (21 < 22) ❌, Jan 4 (25 > 21) ✅, Jan 5 (26 > 25) ✅

## 8. Dates With Higher Temperature Than Previous Date (Self Join approach)

**Difficulty:** 🟡 Medium

**Subtopic:** Self JOIN + DATE functions

---

**Table: temperature**
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| days        | date    |
| temp        | int     |
+-------------+---------+
```

- `days` represents the date.
- `temp` represents the temperature recorded on that date.

---

**Problem**

Write a SQL query to find all **dates where the temperature was higher than the previous day** using a **Self Join** approach instead of LAG.

Return the `days` column only.

**Example 1**

**Input**

temperature table:
```
+------------+------+
| days       | temp |
+------------+------+
| 2024-01-01 | 20   |
| 2024-01-02 | 22   |
| 2024-01-03 | 21   |
| 2024-01-04 | 25   |
| 2024-01-05 | 26   |
+------------+------+
```

**Output**
```
+------------+
| days       |
+------------+
| 2024-01-02 |
| 2024-01-04 |
| 2024-01-05 |
+------------+
```

# 👀 Preview the table

In [ ]:
# 👀 Preview the table
print("temperature table:")
pd.read_sql_query("SELECT * FROM temperature", conn)

temperature table:


,days,temp
0,2024-01-01,20
1,2024-01-02,22
2,2024-01-03,21
3,2024-01-04,25
4,2024-01-05,26


# ✅ Solution using Self Join

In [ ]:
# ✅ Solution using Self Join
query = """
SELECT t1.days
FROM temperature t1
JOIN temperature t2
  ON t2.days = DATE(t1.days, '-1 day')
WHERE t1.temp > t2.temp
"""

pd.read_sql_query(query, conn)

,days
0,2024-01-02
1,2024-01-04
2,2024-01-05


**Explanation**

- We join the `temperature` table with **itself** using aliases `t1` and `t2`.
- `ON t2.days = DATE(t1.days, '-1 day')` matches each date in `t1` with the **previous date** in `t2`.
- `WHERE t1.temp > t2.temp` keeps only rows where today's temperature is higher than yesterday's.
- This gives the **same result as Q7** but uses a self join instead of LAG.
- `DATE(t1.days, '-1 day')` is SQLite syntax — in MySQL use `DATE_SUB(t1.days, INTERVAL 1 DAY)`.

> 💡 **LAG vs Self Join:**
> - LAG is cleaner and more modern — preferred in interviews.
> - Self Join works in older SQL versions that don't support window functions.

## 9. Rolling Average of Posts on Daily Basis Per User

**Difficulty:** 🔴 Hard

**Subtopic:** Window Functions / AVG OVER + ROWS BETWEEN

---

**Table: posts**
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| user_id     | int     |
| date        | date    |
| post_count  | int     |
+-------------+---------+
```

- `user_id` represents the user who made the posts.
- `date` represents the date of posting.
- `post_count` represents the number of posts on that date.

---

**Problem**

Write a SQL query to find the **rolling average of posts** on a daily basis for each `user_id`.

Round the average up to **2 decimal places**.

**Example 1**

**Input**

posts table:
```
+---------+------------+------------+
| user_id | date       | post_count |
+---------+------------+------------+
| 1       | 2024-01-01 | 3          |
| 1       | 2024-01-02 | 5          |
| 1       | 2024-01-03 | 2          |
| 1       | 2024-01-04 | 8          |
| 2       | 2024-01-01 | 1          |
| 2       | 2024-01-02 | 4          |
| 2       | 2024-01-03 | 6          |
+---------+------------+------------+
```

**Output**
```
+---------+------------+------------+-----------------+
| user_id | date       | post_count | rolling_avg     |
+---------+------------+------------+-----------------+
| 1       | 2024-01-01 | 3          | 3.00            |
| 1       | 2024-01-02 | 5          | 4.00            |
| 1       | 2024-01-03 | 2          | 3.33            |
| 1       | 2024-01-04 | 8          | 4.50            |
| 2       | 2024-01-01 | 1          | 1.00            |
| 2       | 2024-01-02 | 4          | 2.50            |
| 2       | 2024-01-03 | 6          | 3.67            |
+---------+------------+------------+-----------------+
```

In [ ]:
# 👀 Preview the table
print("posts table:")
pd.read_sql_query("SELECT * FROM posts", conn)

In [ ]:
# ✅ Solution
query = """
SELECT
    user_id,
    date,
    post_count,
    ROUND(AVG(post_count) OVER (
        PARTITION BY user_id
        ORDER BY date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS rolling_avg
FROM posts
"""

pd.read_sql_query(query, conn)

**Explanation**

- `AVG(post_count) OVER (...)` is a window function that calculates a running average.
- `PARTITION BY user_id` resets the rolling average for each user separately.
- `ORDER BY date` processes rows in chronological order.
- `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` tells SQL to include **all rows from the start up to the current row** in the average calculation.
- `ROUND(..., 2)` rounds the result to 2 decimal places.

> 💡 For a **3-day rolling average** (only last 3 days) you would use:
> `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`

## 10. Rolling Average of Order Amounts

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / AVG OVER + ORDER BY DATE

---

**Table: zomato_orders**
```
+---------------------+---------+
| Column Name         | Type    |
+---------------------+---------+
| user_id             | int     |
| order_id            | int     |
| amount              | int     |
| order_date          | date    |
| delivery_partner_id | int     |
| delivery_rating     | int     |
+---------------------+---------+
```

- `user_id` represents the user who placed the order.
- `amount` represents the order amount.
- `order_date` represents when the order was placed.

---

**Problem**

Write a SQL query to calculate the **rolling average of order amounts** over time across all users.

Return `order_date`, `amount` and `rolling_avg_amount`.

**Example 1**

**Input**

zomato_orders table:
```
+---------+----------+--------+------------+---------------------+-----------------+
| user_id | order_id | amount | order_date | delivery_partner_id | delivery_rating |
+---------+----------+--------+------------+---------------------+-----------------+
| 1       | 101      | 200    | 2024-01-01 | 1                   | 5               |
| 1       | 102      | 300    | 2024-01-03 | 2                   | 4               |
| 2       | 103      | 150    | 2024-01-04 | 1                   | 5               |
| 3       | 104      | 400    | 2024-01-05 | 3                   | 3               |
+---------+----------+--------+------------+---------------------+-----------------+
```

**Output**
```
+------------+--------+--------------------+
| order_date | amount | rolling_avg_amount |
+------------+--------+--------------------+
| 2024-01-01 | 200    | 200.00             |
| 2024-01-03 | 300    | 250.00             |
| 2024-01-04 | 150    | 216.67             |
| 2024-01-05 | 400    | 262.50             |
+------------+--------+--------------------+
```

In [ ]:
# 👀 Preview the table
print("zomato_orders table:")
pd.read_sql_query("SELECT * FROM zomato_orders", conn)

In [ ]:
# ✅ Solution
query = """
SELECT
    order_date,
    amount,
    ROUND(AVG(amount) OVER (
        ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS rolling_avg_amount
FROM zomato_orders
"""

pd.read_sql_query(query, conn)

**Explanation**

- No `PARTITION BY` here — rolling average is calculated **across all orders globally**.
- `ORDER BY order_date` processes orders in chronological order.
- `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` includes all rows from start to current.
- Day 1: avg(200) = 200.00
- Day 2: avg(200, 300) = 250.00
- Day 3: avg(200, 300, 150) = 216.67
- Day 4: avg(200, 300, 150, 400) = 262.50

> 💡 Compare with Q2 — Q2 calculates rolling avg **per user**, Q3 calculates it **across all users globally**.

## 11. Running Total of Order Amounts Over Time

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / SUM OVER + ORDER BY DATE

---

**Table: zomato_orders**
```
+---------------------+---------+
| Column Name         | Type    |
+---------------------+---------+
| user_id             | int     |
| order_id            | int     |
| amount              | int     |
| order_date          | date    |
| delivery_partner_id | int     |
| delivery_rating     | int     |
+---------------------+---------+
```

- `user_id` represents the user who placed the order.
- `amount` represents the order amount.
- `order_date` represents when the order was placed.

---

**Problem**

Write a SQL query to calculate the **running total of order amounts** across all users over time.

Return `order_date`, `user_id`, `amount` and `running_total`.

**Example 1**

**Input**

zomato_orders table:
```
+---------+----------+--------+------------+---------------------+-----------------+
| user_id | order_id | amount | order_date | delivery_partner_id | delivery_rating |
+---------+----------+--------+------------+---------------------+-----------------+
| 1       | 101      | 200    | 2024-01-01 | 1                   | 5               |
| 1       | 102      | 300    | 2024-01-03 | 2                   | 4               |
| 2       | 103      | 150    | 2024-01-04 | 1                   | 5               |
| 3       | 104      | 400    | 2024-01-05 | 3                   | 3               |
+---------+----------+--------+------------+---------------------+-----------------+
```

**Output**
```
+------------+---------+--------+---------------+
| order_date | user_id | amount | running_total |
+------------+---------+--------+---------------+
| 2024-01-01 | 1       | 200    | 200           |
| 2024-01-03 | 1       | 300    | 500           |
| 2024-01-04 | 2       | 150    | 650           |
| 2024-01-05 | 3       | 400    | 1050          |
+------------+---------+--------+---------------+
```

In [ ]:
# 👀 Preview the table
print("zomato_orders table:")
pd.read_sql_query("SELECT * FROM zomato_orders", conn)

In [ ]:
# ✅ Solution
query = """
SELECT
    order_date,
    user_id,
    amount,
    SUM(amount) OVER (
        ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM zomato_orders
"""

pd.read_sql_query(query, conn)

**Explanation**

- `SUM(amount) OVER (ORDER BY order_date)` accumulates the total amount chronologically.
- No `PARTITION BY` — the running total **never resets**, it keeps growing across all users.
- `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` is the explicit window frame — includes everything from the first row up to current.
- Jan 1: 200 → Jan 3: 200+300=500 → Jan 4: 500+150=650 → Jan 5: 650+400=1050.

> 💡 Compare that one resets per user using `PARTITION BY user_id`. This one never resets.